 # 06 · Smoke test: Workbench "New application" flow (doc AI → model → route → Lakebase) + queue read

In [ ]:
%pip install "psycopg[binary]==3.2.3" -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
import json, time, psycopg
from databricks.sdk import WorkspaceClient
w = WorkspaceClient(); out = {}
path = "/Volumes/demo_dnb_catalog/ml_lend_landing/documents/demo/APP-LIVE0001_paystub.pdf"
t = time.time()
txt = spark.sql(f"""SELECT concat_ws('\\n', transform(try_cast(ai_parse_document(content):document:elements AS ARRAY<VARIANT>), e -> try_cast(e:content AS STRING))) AS doc_text
                    FROM read_files('{path}', format => 'binaryFile')""").first()["doc_text"]
out["parse_s"] = round(time.time()-t, 1); t = time.time()
ex = spark.sql("SELECT to_json(ai_extract(:txt, array('employer_name','employee_name','pay_frequency','pay_period_end_date','gross_pay_this_period','year_to_date_gross'))) f", args={"txt": txt}).first()["f"]
ex = json.loads(ex); out["extract_s"] = round(time.time()-t, 1); out["extracted"] = ex
freq = ex["pay_frequency"].lower(); ppm = 26/12 if "bi" in freq and "week" in freq else 2.0 if "semi" in freq else 52/12 if "week" in freq else 1.0
verified = round(float(ex["gross_pay_this_period"].replace("$","").replace(",",""))*ppm); out["verified_monthly"] = verified
out["variance"] = round((7500-verified)/verified, 3)
feat = {"credit_score":702.0,"stated_monthly_income":7500.0,"monthly_debt_payments":900.0,"dti_stated":0.2054,"loan_to_income":0.3556,"requested_amount":32000.0,
        "term_months":60.0,"apr":0.0599,"product_AUTO":1.0,"product_PERSONAL":0.0,"product_CARD":0.0,"product_HELOC":0.0,"channel_DEALER":1.0,
        "channel_ONLINE":0.0,"channel_MOBILE":0.0,"channel_BRANCH":0.0,"member_tenure_years":6.0}
t = time.time()
s = w.api_client.do("POST", "/serving-endpoints/ml-lend-epd-risk/invocations", body={"dataframe_records": [feat]})["predictions"][0]
out["score_s"] = round(time.time()-t, 1); out["score"] = s
out["routing"] = "UNDERWRITER/INCOME_MISMATCH" if out["variance"] > 0.2 else ("UNDERWRITER/HIGH_RISK" if s["risk_band"]=="HIGH" else "AUTO_APPROVE")
EP = "projects/meridianlink-lending/branches/production/endpoints/primary"
host = w.api_client.do("GET", "/api/2.0/postgres/projects/meridianlink-lending/branches/production/endpoints")["endpoints"][0]["status"]["hosts"]["host"]
tok = w.api_client.do("POST", "/api/2.0/postgres/credentials", body={"endpoint": EP})["token"]
with psycopg.connect(host=host, dbname="lending", user=w.current_user.me().user_name, password=tok, sslmode="require", autocommit=True) as c, c.cursor() as cur:
    cur.execute("SELECT count(*), count(*) FILTER (WHERE routing_reason='HIGH_RISK') FROM ml_lend_ml.underwriting_queue_synced WHERE institution_id='CU-SVCU'"); out["synced_queue"] = cur.fetchone()
    cur.execute("SELECT count(*) FROM workbench.live_applications"); out["live_apps"] = cur.fetchone()[0]
    cur.execute("SELECT rolname FROM pg_roles WHERE rolname='85522c3a-a513-4cc7-a183-a8cd4c40a07e'"); out["app_role"] = cur.fetchall()
    cur.execute("""SELECT has_table_privilege('85522c3a-a513-4cc7-a183-a8cd4c40a07e','workbench.underwriting_decisions','INSERT'),
                          has_table_privilege('85522c3a-a513-4cc7-a183-a8cd4c40a07e','ml_lend_ml.underwriting_queue_synced','SELECT')"""); out["app_privs"] = cur.fetchone()
letter = spark.sql("SELECT ai_query('databricks-claude-haiku-5-5', 'Write one sentence greeting a credit union member named Jordan.') l").first()["l"]
out["ai_query_ok"] = bool(letter)
print(json.dumps(out, indent=1, default=str))
dbutils.notebook.exit(json.dumps(out, default=str))

{"parse_s": 23.0, "extract_s": 4.6, "extracted": {"employer_name": "Northwind Logistics", "employee_name": "Jordan Whitaker", "pay_frequency": "Bi-Weekly", "pay_period_end_date": "11/07/2026", "gross_pay_this_period": "$1,938.46", "year_to_date_gross": "$42,646.12"}, "verified_monthly": 4200, "variance": 0.786, "score_s": 0.1, "score": {"epd_probability": 0.030172379908175742, "risk_band": "MEDIUM", "reason_1": "Debt-to-income ratio is high", "reason_2": "Credit score below the level for best pricing", "reason_3": "Requested loan amount"}, "routing": "UNDERWRITER/INCOME_MISMATCH", "synced_queue": [16, 7], "live_apps": 0, "app_role": [["85522c3a-a513-4cc7-a183-a8cd4c40a07e"]], "app_privs": [true, true], "ai_query_ok": true}